# 1. Import of input data

In [ ]:
import nbimporter
import sys
import os
from pathlib import Path

In [ ]:
#Import libraries
try:
    current_dir = RUN_DIR
    print(f"📂 CombinedHeatPower_Turbogas.ipynb executed by RUN.ipynb — using directory: {current_dir}")
except NameError:
    current_dir = Path().resolve()
    print(f"📁 CombinedHeatPower_Turbogas.ipynb executed standalone — using directory: {current_dir}")

Libraries_path = (current_dir / "Libraries.ipynb").resolve()

# If not found, try in the parent levels
if not Libraries_path.exists():
    Libraries_path = (current_dir.parent / "Libraries.ipynb").resolve()

# Run the file
if Libraries_path.exists():
    print(f"📘 Importing libraries from: {Libraries_path}")
    get_ipython().run_line_magic('run', f'"{Libraries_path}"')
else:
    print("⚠️ Libraries.ipynb not found! Some functions might not be available.")

In [ ]:
energysystem = pd.read_excel(energysystem_path, sheet_name = 'CHP-Turbogas')

# 2. Input parameters

In [ ]:
Power = energysystem['Thermal capacity [kW]'];
EfficiencyW = energysystem['Electrical efficiency based on LHV (between 0 and 1)']
EfficiencyQ = energysystem['Heat efficiency based on LHV (between 0 and 1)']
ElectricEnergyproduced = energysystem['Electric energy produced in a year (exported and consumed) [kWh /year]']
ThermalEnergyproduced_primary = energysystem['Thermal energy produced used [kWh /year]'] 
Qused = 1
SupplyTemperature = energysystem['Supply temperature [°C]']
ReturnTemperature = energysystem['Return temperature [°C]']
EnergyInputSourceGasBlend = energysystem['Energy input from gas-blend [kWh/year]']
EnergyInputSourceBiogas = energysystem['Energy input from biogas [kWh/year]']
EnergyInputSourceHydrogen = energysystem['Energy input from hydrogen [kWh/year]']

EquivalentHours = ThermalEnergyproduced_primary / Power

In [ ]:
# Natural gas
LHV = 36.8  # In MJ per Nm3 - Table 3.6 06_V_Natural Gas 3_3 Fuel Data
HHV = 40.4  # In MJ per Nm3 - Table 3.6 06_V_Natural Gas 3_3 Fuel Data

# Biomethane
LHVBiom = 35    # MJ/m3
HHVBiom = 38    # MJ/m3

# Hydrogen
LHVH2 = 120     # MJ/kg
HHVH2 = 142     # MJ/kg 

In [ ]:
Lifespan = np.where(
    Power * (EfficiencyW / EfficiencyQ)  < 25,
    40000 / EquivalentHours,
    100000 / EquivalentHours
)

In [ ]:
EfficiencyTot = EfficiencyW + EfficiencyQ
# Energy produced over the reference study period [kWh]
EnergyProducedLHV = (
    Power
    * EquivalentHours
    * RSP
    / EfficiencyQ        # fuel = heat / thermal efficiency
)

EnergyProducedHHV = EnergyProducedLHV * (HHV / LHV)
TotW = EnergyProducedLHV * EfficiencyW
TotQ = EnergyProducedLHV * EfficiencyQ * Qused

In [ ]:
# MULTIFUNCTIONALITY AND ALLOCATION OF OUTPUTS
# ------------------------------------------------------------------
# Same structure as the RICE notebook.
# Heat and electricity over the RSP, one value per unit (row).
# Electricity is derived from the efficiencies (W/Q = EfficiencyW / EfficiencyQ),
# consistently with the fuel balance in B6.
# ------------------------------------------------------------------
GasTurbine_Q_RSP = np.asarray(ThermalEnergyproduced_primary, dtype=float) * RSP          # kWh_th
GasTurbine_W_RSP = GasTurbine_Q_RSP * (np.asarray(EfficiencyW, dtype=float)
                           / np.asarray(EfficiencyQ, dtype=float))                   # kWh_el

# Consistency check against the electricity declared in the input sheet [kWh/year]
_W_input_RSP = np.asarray(ElectricEnergyproduced, dtype=float) * RSP
_mismatch = np.abs(_W_input_RSP - GasTurbine_W_RSP) > 0.05 * np.maximum(GasTurbine_W_RSP, 1e-9)
if np.any(_mismatch & (_W_input_RSP > 0)):
    print("⚠️ GasTurbine: declared electricity differs by >5% from Q·ηW/ηQ; "
          "allocation uses the efficiency-based value.")

# Rows without supply/return temperatures (both 0 °C): checked BEFORE converting to K
GasTurbine_NoTemperatures = ((np.asarray(SupplyTemperature, dtype=float) == 0)
                       & (np.asarray(ReturnTemperature, dtype=float) == 0))

if Multifunctionality == 'Default':
    # Exergy (Carnot) allocation
    T0_K = float(ExternalTemperature) + 273.15            # local copy, global not modified
    Ts_K = np.asarray(SupplyTemperature, dtype=float) + 273.15
    Tr_K = np.asarray(ReturnTemperature, dtype=float) + 273.15

    with np.errstate(divide='ignore', invalid='ignore'):
        CarnotTemperature = np.where(
            np.isclose(Ts_K, Tr_K),
            Ts_K,                                         # limit of the log-mean for Ts -> Tr
            (Ts_K - Tr_K) / np.log(Ts_K / Tr_K)
        )
        CarnotFactor = 1 - T0_K / CarnotTemperature
        AFQ = (GasTurbine_Q_RSP * CarnotFactor) / (GasTurbine_W_RSP + GasTurbine_Q_RSP * CarnotFactor)

    AFQ_GasTurbine = np.nan_to_num(np.asarray(AFQ, dtype=float).reshape(-1))
    AFW_GasTurbine = 1.0 - AFQ_GasTurbine

else:
    # Power bonus method (EN 15316-4-5): the exported electricity is credited with the
    # primary energy it displaces; the heat takes the remainder of the fuel.
    # Share of the fuel (primary energy) attributed to electricity:
    #   AFW = W * f_el / (F * f_fuel),   AFQ = 1 - AFW
    f_fuel = 1.05   # primary-energy factor of the fuel
    f_el = 2.42     # primary-energy factor of exported electricity
    GasTurbine_F_LHV_RSP = GasTurbine_Q_RSP / np.asarray(EfficiencyQ, dtype=float)   # kWh fuel (LHV)
    AFW_raw = np.asarray((GasTurbine_W_RSP * f_el) / (GasTurbine_F_LHV_RSP * f_fuel), dtype=float).reshape(-1)
    if np.any(AFW_raw > 1):
        print("⚠️ GasTurbine: power bonus exceeds the fuel (AFW > 1); AFW capped at 1, heat burden = 0:",
              np.round(AFW_raw, 3))
    AFW_GasTurbine = np.clip(AFW_raw, 0.0, 1.0)
    AFQ_GasTurbine = 1.0 - AFW_GasTurbine

# No temperatures -> no allocation for that row
AFQ_GasTurbine = np.where(GasTurbine_NoTemperatures, 0.0, AFQ_GasTurbine)
AFW_GasTurbine = np.where(GasTurbine_NoTemperatures, 0.0, AFW_GasTurbine)

In [ ]:
PowerEle = (Power / EfficiencyQ) * EfficiencyW 
RefPowerEle = 10000 #kW
RefMass = 114998.5 #kg
mass = PowerEle * (RefMass / RefPowerEle)

# 3. Life cycle inventories

## 3.1 Module A1-3

In [ ]:
GasTurbine_concreteAT = ((0.22 * mass) / 2400) * AFQ_GasTurbine
GasTurbine_concreteCH = ((0.41 * mass) / 2400) * AFQ_GasTurbine
GasTurbine_copper = (0.03 * mass) * AFQ_GasTurbine
GasTurbine_diesel = (np.full(Power.shape, 759000)) * AFQ_GasTurbine
GasTurbine_electricity = (np.full(Power.shape, 46900)) * AFQ_GasTurbine
GasTurbine_heat = (np.full(Power.shape, 721050)) * AFQ_GasTurbine
GasTurbine_PEHD = (0.08 * mass) * AFQ_GasTurbine
GasTurbine_reinforcingsteel = (0.25 * mass) * AFQ_GasTurbine
GasTurbine_chromiumsteel = (0.01 * mass) * AFQ_GasTurbine

## 3.2 Module A4

In [ ]:
GasTurbine_Truk_16_32 = ((500 + 200)*(mass)/1000) * AFQ_GasTurbine
GasTurbine_Light_Truk = (50*(mass)/1000) * AFQ_GasTurbine

## 3.3 Module B2

In [ ]:
GasTurbine_LightTruk_B2 = (50 * (Lifespan/2) * 0.01 * (mass/1000))* AFQ_GasTurbine

## 3.4 Module B4

In [ ]:
GasTurbineNumberReplacements = (np.ceil(np.divide(ReqSL, Lifespan) - 1))
SFGasTurbine = (RSP / ReqSL)  # Scaling factor

## 3.5 Module B6

In [ ]:
#Parameters for energy input source repartition
GasTurbine_total_energy_input = (
    EnergyInputSourceGasBlend
    + EnergyInputSourceBiogas
    + EnergyInputSourceHydrogen
)

GasBlend = (
    EnergyInputSourceGasBlend
    / GasTurbine_total_energy_input
)

Biogas = (
    EnergyInputSourceBiogas
    / GasTurbine_total_energy_input
)

Hydrogen = (
    EnergyInputSourceHydrogen
    / GasTurbine_total_energy_input
)

In [ ]:
# ------------------------------------------------------------------
# B6 - Gas Turbine
# Fuel balances and direct emissions are expressed on an LHV basis.
# ------------------------------------------------------------------

# Useful thermal output over the reference study period [MJ heat]
GasTurbine_heat_output_MJ = (
    np.asarray(ThermalEnergyproduced_primary, dtype=float)
    * RSP
    * 3.6
)

# Total CHP fuel input required for the useful heat [MJ fuel, LHV]
GasTurbine_total_fuel_input_MJ = (
    GasTurbine_heat_output_MJ
    / EfficiencyQ
)

# Fuel input allocated to the thermal co-product [MJ fuel, LHV]
GasTurbine_fuel_energy_MJin = (
    GasTurbine_total_fuel_input_MJ
    * AFQ_GasTurbine
)

# ------------------------------------------------------------------
# Fuel inputs by origin [MJ fuel, LHV]
# ------------------------------------------------------------------

# Gas blend
GasTurbine_GasBlend_NG_MJin = (
    GasTurbine_fuel_energy_MJin
    * GasBlend
    * A
)

GasTurbine_GasBlend_Biomethane_MJin = (
    GasTurbine_fuel_energy_MJin
    * GasBlend
    * B
)

GasTurbine_GasBlend_Hydrogen_MJin = (
    GasTurbine_fuel_energy_MJin
    * GasBlend
    * C
)

# Standalone biomethane
# Legacy variable "Biogas" is treated as biomethane.
GasTurbine_Biomethane_MJin = (
    GasTurbine_fuel_energy_MJin
    * Biogas
)

# Standalone hydrogen
GasTurbine_Hydrogen_MJin = (
    GasTurbine_fuel_energy_MJin
    * Hydrogen
)

# ------------------------------------------------------------------
# Inputs to ecoinvent
# ------------------------------------------------------------------

# Natural gas [m3]
GasTurbine_GasBlend_NG_Input = (
    GasTurbine_GasBlend_NG_MJin
    / LHV
)

# Biomethane [m3]
GasTurbine_GasBlend_Biomethane_Input = (
    GasTurbine_GasBlend_Biomethane_MJin
    / LHVBiom
)

GasTurbine_Biomethane_Input = (
    GasTurbine_Biomethane_MJin
    / LHVBiom
)

# Hydrogen:
# MJ_H2 -> kWh_H2 -> kg_H2 -> kWh electricity from photovoltaic system
GasTurbine_LHVH2_kWh_kg = (
    LHVH2
    / 3.6
)

GasTurbine_GasBlend_Hydrogen_kWhH2 = (
    GasTurbine_GasBlend_Hydrogen_MJin
    / 3.6
)

GasTurbine_GasBlend_Hydrogen_kg = (
    GasTurbine_GasBlend_Hydrogen_kWhH2
    / GasTurbine_LHVH2_kWh_kg
)

GasTurbine_GasBlend_Hydrogen_Input = (
    GasTurbine_GasBlend_Hydrogen_kg
    * 55
)

GasTurbine_Hydrogen_kWhH2 = (
    GasTurbine_Hydrogen_MJin
    / 3.6
)

GasTurbine_Hydrogen_kg = (
    GasTurbine_Hydrogen_kWhH2
    / GasTurbine_LHVH2_kWh_kg
)

GasTurbine_Hydrogen_Input = (
    GasTurbine_Hydrogen_kg
    * 55
)

# ------------------------------------------------------------------
# Fuel-energy aggregates for direct emissions [MJin, LHV]
# ------------------------------------------------------------------

GasTurbine_non_hydrogen_energy_MJin = (
    GasTurbine_GasBlend_NG_MJin
    + GasTurbine_GasBlend_Biomethane_MJin
    + GasTurbine_Biomethane_MJin
)

GasTurbine_fossil_energy_MJin = (
    GasTurbine_GasBlend_NG_MJin
)

GasTurbine_biogenic_energy_MJin = (
    GasTurbine_GasBlend_Biomethane_MJin
    + GasTurbine_Biomethane_MJin
)

# Available for future H2-specific direct-emission factors
GasTurbine_hydrogen_energy_MJin = (
    GasTurbine_GasBlend_Hydrogen_MJin
    + GasTurbine_Hydrogen_MJin
)

# ------------------------------------------------------------------
# Direct emissions
# Factors are per MJ fuel input, LHV basis.
# ------------------------------------------------------------------

GasTurbine_WaterCooling = (
    0.00618146537053705
    * GasTurbine_non_hydrogen_energy_MJin
)

GasTurbine_Acetaldehyde = (
    0.000000001
    * GasTurbine_non_hydrogen_energy_MJin
)

GasTurbine_AceticAcid = (
    0.00000015
    * GasTurbine_non_hydrogen_energy_MJin
)

GasTurbine_Benzene = (
    0.00000039
    * GasTurbine_non_hydrogen_energy_MJin
)

GasTurbine_Benzoapyrene = (
    0.00000000001
    * GasTurbine_non_hydrogen_energy_MJin
)

GasTurbine_Butane = (
    0.00000068
    * GasTurbine_non_hydrogen_energy_MJin
)

GasTurbine_CarbonDioxideFossil = (
    0.05406
    * GasTurbine_fossil_energy_MJin
)

GasTurbine_CarbonDioxideBiogenic = (
    0.05406
    * GasTurbine_biogenic_energy_MJin
)

GasTurbine_CarbonMonoxideFossil = (
    0.000003
    * GasTurbine_non_hydrogen_energy_MJin
)

GasTurbine_DinitrogenMonoxide = (
    0.00000048
    * GasTurbine_non_hydrogen_energy_MJin
)

GasTurbine_Formaldehyde = (
    0.00000010
    * GasTurbine_non_hydrogen_energy_MJin
)

GasTurbine_Mercury = (
    0.00000000003
    * GasTurbine_non_hydrogen_energy_MJin
)

GasTurbine_MethaneFossil = (
    0.00000193
    * GasTurbine_non_hydrogen_energy_MJin
)

GasTurbine_NitrogenOxides = (
    0.00001931
    * GasTurbine_non_hydrogen_energy_MJin
)

GasTurbine_PAH = (
    0.00000001
    * GasTurbine_non_hydrogen_energy_MJin
)

GasTurbine_Particulates = (
    0.00000010
    * GasTurbine_non_hydrogen_energy_MJin
)

GasTurbine_Pentane = (
    0.00000116
    * GasTurbine_non_hydrogen_energy_MJin
)

GasTurbine_Propane = (
    0.00000019
    * GasTurbine_non_hydrogen_energy_MJin
)

GasTurbine_PropionicAcid = (
    0.00000002
    * GasTurbine_non_hydrogen_energy_MJin
)

GasTurbine_SulfurDioxide = (
    0.00000048
    * GasTurbine_non_hydrogen_energy_MJin
)

GasTurbine_Toluene = (
    0.00000019
    * GasTurbine_non_hydrogen_energy_MJin
)

GasTurbine_WaterAir = (
    0.0000729530701641572
    * GasTurbine_non_hydrogen_energy_MJin
)

GasTurbine_WaterWater = (
    0.00610851230037289
    * GasTurbine_non_hydrogen_energy_MJin
)

## 3.6 Module C2

In [ ]:
GasTurbine_Transport_to_treatment_plant = 50 * (mass/1000) * AFQ_GasTurbine

## 3.7 Module C3

In [ ]:
GasTurbine_EOL_concreteAT = GasTurbine_concreteAT
GasTurbine_EOL_concreteCH = GasTurbine_concreteCH
GasTurbine_EOL_copper = GasTurbine_copper
GasTurbine_EOL_PEHD = GasTurbine_PEHD
GasTurbine_EOL_reinforcingsteel = GasTurbine_reinforcingsteel
GasTurbine_EOL_chromiumsteel = GasTurbine_chromiumsteel

## 3.8 Module C4

In [ ]:
GasTurbine_EOL_landfill_concreteAT = GasTurbine_EOL_concreteAT * LandfillConcrete
GasTurbine_EOL_landfill_concreteCH = GasTurbine_EOL_concreteCH * LandfillConcrete
GasTurbine_EOL_landfill_copper = GasTurbine_EOL_copper * LandfillMetals
GasTurbine_EOL_inc_PEHD = GasTurbine_EOL_PEHD * IncinerationPEHD
GasTurbine_EOL_landfill_PEHD = GasTurbine_EOL_PEHD * LandfillPEHD
GasTurbine_EOL_landfill_reinforcingsteel = GasTurbine_EOL_reinforcingsteel * LandfillMetals
GasTurbine_EOL_landfill_chromiumsteel = GasTurbine_EOL_chromiumsteel * LandfillMetals2

## 3.9 Module D1 (related with the export of secondary materials)

In [ ]:
GasTurbine_EOL_recycling_concreteAT = (RecyclingConcrete * RecyclingEfficiencyConcrete * GasTurbine_EOL_concreteAT) - (RecycledContentConcrete * GasTurbine_EOL_concreteAT)
GasTurbine_EOL_primary_concreteAT = ((RecycledContentConcrete * GasTurbine_EOL_concreteAT) - (RecyclingConcrete * RecyclingEfficiencyConcrete * GasTurbine_EOL_concreteAT)) * SRinert

GasTurbine_EOL_recycling_concreteCH = (RecyclingConcrete * RecyclingEfficiencyConcrete * GasTurbine_EOL_concreteCH) - (RecycledContentConcrete * GasTurbine_EOL_concreteCH)
GasTurbine_EOL_primary_concreteCH = ((RecycledContentConcrete * GasTurbine_EOL_concreteCH) - (RecyclingConcrete * RecyclingEfficiencyConcrete * GasTurbine_EOL_concreteCH)) * SRinert

GasTurbine_EOL_recycling_copper = (RecyclingMetals * RecyclingEfficiencyMetals * GasTurbine_EOL_copper) - (RecycledContentMetals * GasTurbine_EOL_copper)
GasTurbine_EOL_primary_copper = ((RecycledContentMetals * GasTurbine_EOL_copper) - (RecyclingMetals * RecyclingEfficiencyMetals * GasTurbine_EOL_copper)) * SRmetals

GasTurbine_EOL_recycling_HDPE = (RecyclingPEHD * RecyclingEfficiencyPlastics * GasTurbine_EOL_PEHD) - (RecycledContentPlastic * GasTurbine_EOL_PEHD)
GasTurbine_EOL_primary_HDPE = ((RecycledContentPlastic * GasTurbine_EOL_PEHD) - (RecyclingPEHD * RecyclingEfficiencyPlastics * GasTurbine_EOL_PEHD)) * SRinert

GasTurbine_EOL_recycling_reinforcingsteel = (RecyclingMetals * RecyclingEfficiencyMetals * GasTurbine_EOL_reinforcingsteel) - (RecycledContentMetals * GasTurbine_EOL_reinforcingsteel)
GasTurbine_EOL_primary_reinforcingsteel = ((RecycledContentMetals * GasTurbine_EOL_reinforcingsteel) - (RecyclingMetals * RecyclingEfficiencyMetals * GasTurbine_EOL_reinforcingsteel)) * SRmetals

GasTurbine_EOL_recycling_chromiumsteel = (RecyclingMetals2 * RecyclingEfficiencyMetals * GasTurbine_EOL_chromiumsteel) - (RecycledContentMetals * GasTurbine_EOL_chromiumsteel)
GasTurbine_EOL_primary_chromiumsteel = ((RecycledContentMetals * GasTurbine_EOL_chromiumsteel) - (RecyclingMetals2 * RecyclingEfficiencyMetals * GasTurbine_EOL_chromiumsteel)) * SRmetals

## 3.10 Module D3 (related to the export of energy as result of waste incineration)

In [ ]:
GasTurbine_EOL_energyrecoveryheat_PEHD = - GasTurbine_EOL_inc_PEHD * LHVHDPE * Xincheat
GasTurbine_EOL_energyrecoveryele_PEHD = (- GasTurbine_EOL_inc_PEHD * LHVHDPE * Xincele) / 3.6

# 4. Data for export

In [ ]:
# Useful thermal energy produced by the Gas Turbine
# over the reference study period [kWh]
ThermalEnergyProduced_GasTurbine = (
    np.asarray(ThermalEnergyproduced_primary, dtype=float)
    * RSP
)

# Electricity produced over the reference study period [kWh_el]
ElectricEnergyProduced_GasTurbine = GasTurbine_W_RSP